# Agent 365 Lander (optional source)

Lands the **Agents 365** registry export into the Lakehouse Delta table `dbo.agents_365`, which the
Fabric dashboard reads via `FabricTable("agents_365")`.

Keeping Agent 365 in the Lakehouse (rather than a SharePoint URL in the report) keeps the Fabric
model **100% Lakehouse-sourced** — no gateway, no privacy-firewall, Direct Lake eligible.

**How to use:** drop the Agents 365 CSV at `Files/agent365/agents.csv` (or point `SOURCE_CSV` at a
OneLake shortcut), attach this notebook to the `CopilotAdoptionLake` Lakehouse, and Run all.
If the file is absent the dashboard's Agents 365 table simply loads empty (it is an optional source).

In [ ]:
# === CONFIG ===
SOURCE_CSV   = 'Files/agent365/agents.csv'   # drop the Agents 365 (MAC) export here, or a OneLake shortcut path
OUTPUT_TABLE = 'dbo.agents_365'              # Delta table read by the dashboard's Agents 365 table
WRITE_MODE   = 'overwrite'                   # 'overwrite' for full snapshots; 'append' for incremental

In [ ]:
# === LAND CSV -> Delta (normalize old/new headers to the canonical dashboard schema) ===
import notebookutils
from pyspark.sql import functions as F

def _exists(path):
    try:
        d = '/'.join(path.split('/')[:-1])
        return any(fi.name == path.split('/')[-1] for fi in notebookutils.fs.ls(d))
    except Exception:
        return False

if not _exists(SOURCE_CSV):
    print(f"Agent 365 export not found at {SOURCE_CSV} \u2014 nothing landed. "
          f"The dashboard's Agents 365 table will load empty (optional source).")
else:
    df = (spark.read
          .option('header', True)
          .option('multiLine', True)
          .option('escape', '"')
          .option('encoding', 'UTF-8')
          .csv(SOURCE_CSV))

    def _rename_first(frame, target, candidates):
        for candidate in candidates:
            if candidate in frame.columns:
                return frame if candidate == target else frame.withColumnRenamed(candidate, target)
        return frame

    rename_plan = [
        ('Agent name', ['Agent name', 'Agent Name', 'Name', 'name']),
        ('Supported in', ['Supported in', 'Channel']),
        ('Agent creator', ['Agent creator', 'Publisher', 'Owner', 'Developer Name']),
        ('Agent type (A365)', ['Agent type (A365)', 'Publisher Type', 'Type']),
        ('Agent creator ID', ['Agent creator ID', 'Creator Id', 'Creator ID', 'Owner', 'Created by']),
        ('Agent description', ['Agent description', 'Description']),
        ('Created in', ['Created in', 'Platform']),
        ('Last updated', ['Last updated', 'Last Modified', 'Last modified']),
    ]

    for target, candidates in rename_plan:
        df = _rename_first(df, target, candidates)

    canonical_cols = [
        'Agent name', 'Supported in', 'Date created', 'Agent creator', 'Agent type (A365)',
        'Version', 'Availability', 'Agent creator ID', 'Agent description', 'Created in',
        'Last updated', 'Custom actions', 'Title ID', 'Sensitivity',
        'Can read OneDrive and Sharepoint items', 'OneDrive and Sharepoint items',
        'Can read OneDrive files', 'OneDrive files', 'OneDrive sites',
        'Can read Sharepoint sites and files', 'Sharepoint files', 'Sharepoint sites',
        'Can extend to Graph connector', 'Graph connector details',
        'Can generate images using user prompt', 'Can use code interpreter',
        'Contains uploaded files', 'Uploaded files', 'Status'
    ]

    for col_name in canonical_cols:
        if col_name not in df.columns:
            df = df.withColumn(col_name, F.lit(None).cast('string'))

    ordered = canonical_cols + [c for c in df.columns if c not in canonical_cols]
    df = df.select(*ordered)

    print(f'agents_365 rows: {df.count():,} | columns: {len(df.columns)}')
    (df.write.mode(WRITE_MODE)
       .option('overwriteSchema', 'true')
       .option('delta.columnMapping.mode', 'name')
       .format('delta').saveAsTable(OUTPUT_TABLE))
    print(f'wrote -> {OUTPUT_TABLE} ({WRITE_MODE})')